# Week 1: Data Ingestion & Cleaning
This notebook processes the raw monthly P2M transaction statistics from NPCI, detects the headers dynamically, maps category aliases to standardized labels to handle label drift, and compiles the files into a single cleaned CSV.

In [1]:
import os
import re
import pandas as pd
import numpy as np
import datetime
import openpyxl

In [2]:
RAW_DIR = '../data/raw'
PROCESSED_DIR = '../data/processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)

MONTH_MAPPING = {
    'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'may': 5, 'jun': 6,
    'jul': 7, 'aug': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12
}

def standardize_category(raw_cat):
    cat = str(raw_cat).strip().lower()
    if 'grocery' in cat or 'supermarket' in cat:
        return 'Grocery & Supermarkets'
    if 'eating' in cat or 'restaurant' in cat or 'food' in cat or 'bakery' in cat or 'drinking' in cat:
        return 'Food & Beverages'
    if 'travel' in cat or 'transport' in cat or 'hotel' in cat or 'service station' in cat or 'railway' in cat:
        return 'Travel & Transport'
    if 'utility' in cat or 'utilities' in cat or 'bill payment' in cat or 'telecom' in cat or 'cable' in cat:
        return 'Utilities & Bill Payment'
    if 'education' in cat or 'school' in cat or 'college' in cat or 'admission' in cat or 'edtech' in cat:
        return 'Education'
    if 'insurance' in cat or 'financial' in cat or 'securities' in cat or 'broker' in cat:
        return 'Insurance & Financial Services'
    if 'healthcare' in cat or 'pharmacy' in cat or 'drug store' in cat or 'hospital' in cat:
        return 'Healthcare & Pharmacy'
    return cat.title()

In [3]:
def find_header_and_read(filepath):
    df_raw = pd.read_excel(filepath, header=None)
    
    header_row_idx = None
    for idx, row in df_raw.iterrows():
        row_str = [str(x).lower() for x in row.values]
        
        # Skip rows with less than 3 non-null values (like titles)
        non_null_count = sum(1 for x in row.values if pd.notna(x) and str(x).strip() != '')
        if non_null_count < 3:
            continue
            
        has_cat = any(any(k in s for k in ['category', 'segment', 'mcc', 'description']) for s in row_str if isinstance(s, str))
        has_vol = any(any(k in s for k in ['volume', 'txn count', 'no. of txn', 'txns']) for s in row_str if isinstance(s, str))
        has_val = any(any(k in s for k in ['value', 'amount', 'val']) for s in row_str if isinstance(s, str))
        
        if has_cat and (has_vol or has_val):
            header_row_idx = idx
            break
            
    if header_row_idx is None:
        raise ValueError(f'Could not detect headers in {filepath}')
        
    headers = df_raw.iloc[header_row_idx].astype(str).tolist()
    headers = [h.strip().replace('\n', ' ') for h in headers]
    
    df_data = df_raw.iloc[header_row_idx + 1:].copy()
    df_data.columns = headers
    df_data = df_data.dropna(how='all')
    return df_data

In [4]:
def process_file(filepath, filename):
    # Parse real filename format: Ecosystem-Statistics-UPI-Merchant-category-classification-YYYY-Mon.xlsx
    match_real = re.search(r'Ecosystem-Statistics-UPI-Merchant-category-classification-(\d{4})-([A-Za-z]{3})', filename, re.IGNORECASE)
    if match_real:
        year = int(match_real.group(1))
        month_str = match_real.group(2).lower()
        month = MONTH_MAPPING[month_str]
        month_date = datetime.datetime(year, month, 1)
    else:
        # Fallback to test format: upi_ecosystem_stats_YYYY_MM.xlsx
        match_test = re.search(r'upi_ecosystem_stats_(\d{4})_(\d{2})', filename)
        if not match_test:
            raise ValueError(f'Filename {filename} does not match any expected pattern')
        year = int(match_test.group(1))
        month = int(match_test.group(2))
        month_date = datetime.datetime(year, month, 1)
    
    df = find_header_and_read(filepath)
    
    cat_col = None
    vol_col = None
    val_col = None
    
    for col in df.columns:
        col_lower = str(col).lower()
        if re.search(r'(category|segment|mcc.*desc|mcc description|description)', col_lower):
            cat_col = col
        elif re.search(r'(volume|txn.*count|no.*of.*txn|txns)', col_lower):
            vol_col = col
        elif re.search(r'(value|amount|val)', col_lower):
            val_col = col
            
    if not cat_col or not vol_col or not val_col:
        raise ValueError(f'Columns missing in {filename}. Columns found: {df.columns.tolist()}')
        
    df = df.dropna(subset=[cat_col])
    df = df[~df[cat_col].astype(str).str.lower().str.contains(r'(total|grand|overall|aggregate|s\.no|sr\.)')]
    
    df = df[[cat_col, vol_col, val_col]].copy()
    df.columns = ['raw_category', 'raw_volume', 'raw_value']
    
    df['raw_volume'] = df['raw_volume'].astype(str).str.replace(',', '').astype(float)
    df['raw_value'] = df['raw_value'].astype(str).str.replace(',', '').astype(float)
    
    df['volume'] = (df['raw_volume'] * 1_000_000).astype(np.int64)
    df['value_cr'] = df['raw_value'].astype(np.float64)
    df['category'] = df['raw_category'].apply(standardize_category)
    
    df['month'] = month_date
    df['avg_ticket'] = (df['value_cr'] * 10_000_000) / df['volume']
    
    return df[['month', 'category', 'volume', 'value_cr', 'avg_ticket']]

In [5]:
all_dfs = []
for file in os.listdir(RAW_DIR):
    if file.endswith('.xlsx') or file.endswith('.xls'):
        filepath = os.path.join(RAW_DIR, file)
        try:
            df_cleaned = process_file(filepath, file)
            all_dfs.append(df_cleaned)
            print(f'Ingested {file}')
        except Exception as e:
            print(f'Error on {file}: {e}')

if not all_dfs:
    raise ValueError('No files parsed successfully')

master_df = pd.concat(all_dfs, ignore_index=True)

# Group and aggregate by month and category to sum sub-categories
master_df = master_df.groupby(['month', 'category']).agg({
    'volume': 'sum',
    'value_cr': 'sum'
}).reset_index()

# Recalculate avg_ticket after aggregation
master_df['avg_ticket'] = (master_df['value_cr'] * 10_000_000) / master_df['volume']

master_df = master_df.sort_values(by=['month', 'category']).reset_index(drop=True)

master_filepath = os.path.join(PROCESSED_DIR, 'master.csv')
master_df.to_csv(master_filepath, index=False)
print(f'Created master.csv with {len(master_df)} rows.')
master_df.head()

C:\Users\saikr\AppData\Local\Temp\ipykernel_12752\2458558954.py:37: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df[~df[cat_col].astype(str).str.lower().str.contains(r'(total|grand|overall|aggregate|s\.no|sr\.)')]
C:\Users\saikr\AppData\Local\Temp\ipykernel_12752\2458558954.py:37: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df[~df[cat_col].astype(str).str.lower().str.contains(r'(total|grand|overall|aggregate|s\.no|sr\.)')]
C:\Users\saikr\AppData\Local\Temp\ipykernel_12752\2458558954.py:37: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df[~df[cat_col].astype(str).str.lower().str.contains(r'(total|grand|overall|aggregate|s\.no|sr\.)')]
C:\Users\saikr\AppData\Local\Temp\ipykernel_12752\2458558954.py:37: UserW

Ingested Ecosystem-Statistics-UPI-Merchant-category-classification-2022-Apr.xlsx
Ingested Ecosystem-Statistics-UPI-Merchant-category-classification-2022-Aug.xlsx
Ingested Ecosystem-Statistics-UPI-Merchant-category-classification-2022-Dec.xlsx
Ingested Ecosystem-Statistics-UPI-Merchant-category-classification-2022-Feb.xlsx
Ingested Ecosystem-Statistics-UPI-Merchant-category-classification-2022-Jan.xlsx
Ingested Ecosystem-Statistics-UPI-Merchant-category-classification-2022-Jul.xlsx
Ingested Ecosystem-Statistics-UPI-Merchant-category-classification-2022-Jun.xlsx


C:\Users\saikr\AppData\Local\Temp\ipykernel_12752\2458558954.py:37: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df[~df[cat_col].astype(str).str.lower().str.contains(r'(total|grand|overall|aggregate|s\.no|sr\.)')]
C:\Users\saikr\AppData\Local\Temp\ipykernel_12752\2458558954.py:37: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df[~df[cat_col].astype(str).str.lower().str.contains(r'(total|grand|overall|aggregate|s\.no|sr\.)')]
C:\Users\saikr\AppData\Local\Temp\ipykernel_12752\2458558954.py:37: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df[~df[cat_col].astype(str).str.lower().str.contains(r'(total|grand|overall|aggregate|s\.no|sr\.)')]
C:\Users\saikr\AppData\Local\Temp\ipykernel_12752\2458558954.py:37: UserW

Ingested Ecosystem-Statistics-UPI-Merchant-category-classification-2022-Mar.xlsx
Ingested Ecosystem-Statistics-UPI-Merchant-category-classification-2022-May.xlsx
Ingested Ecosystem-Statistics-UPI-Merchant-category-classification-2022-Nov.xlsx
Ingested Ecosystem-Statistics-UPI-Merchant-category-classification-2022-Oct.xlsx
Ingested Ecosystem-Statistics-UPI-Merchant-category-classification-2022-Sep.xlsx
Ingested Ecosystem-Statistics-UPI-Merchant-category-classification-2023-Apr.xlsx
Ingested Ecosystem-Statistics-UPI-Merchant-category-classification-2023-Aug.xlsx
Ingested Ecosystem-Statistics-UPI-Merchant-category-classification-2023-Dec.xlsx
Ingested Ecosystem-Statistics-UPI-Merchant-category-classification-2023-Feb.xlsx
Ingested Ecosystem-Statistics-UPI-Merchant-category-classification-2023-Jan.xlsx
Ingested Ecosystem-Statistics-UPI-Merchant-category-classification-2023-Jul.xlsx
Ingested Ecosystem-Statistics-UPI-Merchant-category-classification-2023-Jun.xlsx
Ingested Ecosystem-Statistic

C:\Users\saikr\AppData\Local\Temp\ipykernel_12752\2458558954.py:37: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df[~df[cat_col].astype(str).str.lower().str.contains(r'(total|grand|overall|aggregate|s\.no|sr\.)')]
C:\Users\saikr\AppData\Local\Temp\ipykernel_12752\2458558954.py:37: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df[~df[cat_col].astype(str).str.lower().str.contains(r'(total|grand|overall|aggregate|s\.no|sr\.)')]
C:\Users\saikr\AppData\Local\Temp\ipykernel_12752\2458558954.py:37: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df[~df[cat_col].astype(str).str.lower().str.contains(r'(total|grand|overall|aggregate|s\.no|sr\.)')]
C:\Users\saikr\AppData\Local\Temp\ipykernel_12752\2458558954.py:37: UserW

...

C:\Users\saikr\AppData\Local\Temp\ipykernel_12752\2458558954.py:37: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df[~df[cat_col].astype(str).str.lower().str.contains(r'(total|grand|overall|aggregate|s\.no|sr\.)')]


Ingested Ecosystem-Statistics-UPI-Merchant-category-classification-2026-Feb.xlsx


Ingested Ecosystem-Statistics-UPI-Merchant-category-classification-2026-Jan.xlsx
Ingested Ecosystem-Statistics-UPI-Merchant-category-classification-2026-Mar.xlsx
Ingested Ecosystem-Statistics-UPI-Merchant-category-classification-2026-May.xlsx
Created master.csv with 1202 rows.


C:\Users\saikr\AppData\Local\Temp\ipykernel_12752\2458558954.py:37: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df[~df[cat_col].astype(str).str.lower().str.contains(r'(total|grand|overall|aggregate|s\.no|sr\.)')]
C:\Users\saikr\AppData\Local\Temp\ipykernel_12752\2458558954.py:37: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df[~df[cat_col].astype(str).str.lower().str.contains(r'(total|grand|overall|aggregate|s\.no|sr\.)')]
C:\Users\saikr\AppData\Local\Temp\ipykernel_12752\2458558954.py:37: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df[~df[cat_col].astype(str).str.lower().str.contains(r'(total|grand|overall|aggregate|s\.no|sr\.)')]


,month,category,volume,value_cr,avg_ticket
0,2022-01-01,Bakeries,13480000,300.66,223.041543
1,2022-01-01,Business Services Not Elsewhere Classified,32369999,2942.86,909.131940
2,2022-01-01,Caterers,12290000,368.74,300.032547
3,2022-01-01,Debit Card To Wallet Credit (Wallet Top Up),36190000,3598.43,994.316109
4,2022-01-01,Debt Collection Agencies,39910000,30589.20,7664.545227
